In [0]:
import sys
import os

project_root = os.path.abspath(os.path.join(os.getcwd(), "../.."))

if project_root not in sys.path:
    sys.path.append(project_root)

from modules.utils.date_utils import get_month_start_n_months_ago
from pyspark.sql.functions import date_format

In [0]:
#get the first day of the month two months ago
two_months_ago = get_month_start_n_months_ago(2)


In [0]:
#read the 'yellow_trips_enriched' table from the nyctax.`02_silver`.yellow_trips_enriched schema
#and filter to only include trips with a pickup datetime
#later than the start date from two months ago

df = spark.read.table("nyctax.`02_silver`.yellow_trips_enriched").filter(
    f"tpep_pickup_datetime > '{two_months_ago}'"
)

In [0]:
#add a year_month column, formatted yyyy-MM
df = df.withColumn("year_month", date_format("tpep_pickup_datetime", "yyyy-MM"))

In [0]:
df.write.\
    option("path", "abfss://nyctaxi-yellow@nyctaxistorage118.dfs.core.windows.net/yellow_trips_export").\
    format("json").\
    mode("overwrite").\
    partitionBy("vendor", "year_month"). \
    saveAsTable("nyctax.04_export.yellow_trips_export")